# Model Training with Leakage

Window-level random split (the leakage) on the **same fixed pipeline and the
same pre-specified configs** as the honest LORO run. Nothing is tuned — every
config is run and reported. Segment-level metrics only: a shuffled split has no
intact record timeline, so event-level SzCORE is not computable here by design.

Two knobs at the top of the CONFIG cell:
- `USE_ALL_FILES` : `False` = seizure-files-only (faithful reproduction of the
  cited protocol). `True` = all files incl. interictal (matched population vs honest).
- `PATIENTS` : one patient now, the full list later — no other change needed.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, accuracy_score, confusion_matrix,
)

## Load one patient

Always loads BOTH seizure and non-seizure files; whether the interictal-only
files actually enter the pool is decided later by `USE_ALL_FILES`. Mirrors the
honest `load_patient` (same complex-window handling, same 1530-orientation fix).

In [4]:
N_FEATURES = 1530
IMAG_TOL   = 1e-6   # |imag| above this = invalid GC window, dropped (as in honest run)

SEIZURE_DIR     = Path("../Features/Seizure")
NON_SEIZURE_DIR = Path("../Features/Non_Seizure")


def _read_feats(h, key="all_feats"):
    """Read features, handling MATLAB complex compound dtype. Returns
    (X real, rows=windows) and a bad-window mask (|imag| too large)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    """Return seizure_records and non_records dicts: {record_name: (X, y)}."""
    records = {}

    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        with h5py.File(f, "r") as h:
            X, bad = _read_feats(h, "all_feats")
            yv = h["all_labels"][:].squeeze()
        y = (yv if yv.shape[0] == X.shape[0] else yv[: X.shape[0]]).astype(int)
        if bad.any():
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)
        print("[SEIZURE]    ", f.stem, X.shape, "positives =", int(y.sum()))

    for f in sorted(NON_SEIZURE_DIR.glob(f"{patient}*_features.mat")):
        with h5py.File(f, "r") as h:
            X, bad = _read_feats(h, "all_feats")
        if bad.any():
            X = X[~bad]
        y = np.zeros(X.shape[0], dtype=int)
        records[f.stem.replace("_features", "")] = (X, y)
        print("[NON-SEIZURE]", f.stem, X.shape, "positives = 0")

    seizure_records = {k: v for k, v in records.items() if v[1].sum() > 0}
    non_records     = {k: v for k, v in records.items() if v[1].sum() == 0}
    return seizure_records, non_records

## Matched pipeline

Byte-for-byte identical to the honest run's `make_pipe`: `StandardScaler` ->
`SelectPercentile(f_classif, 10)` -> RBF-SVM with `class_weight={0:1, 1:100}`.
The ONLY thing that differs between this notebook and the honest one is the
split (random windows vs LORO). Do not change anything here.

In [5]:
def make_pipe(cfg):
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif, percentile=cfg["percentile"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight='balanced', cache_size=500)),
    ])

## CONFIG + RUN

Same nine pre-specified configs as the honest run (C x gamma, percentile fixed
at 10). Every config is run on every patient and every row is reported — nothing
is selected. For each (patient, config): pool windows, take one stratified
random split (**this shuffle is the leakage**), fit on train, score test once,
record segment metrics.

In [6]:

def build_pool(seizure_records, non_records):
    """Patient-specific window pool. Seizure files always; interictal files only
    when USE_ALL_FILES. Train and test are drawn from the SAME pool (internally
    consistent population)."""
    pool = dict(seizure_records)
    if USE_ALL_FILES:
        pool.update(non_records)
    X = np.vstack([X for X, _ in pool.values()])
    y = np.hstack([y for _, y in pool.values()]).astype(int)
    return X, y


def segment_metrics(y_true, scores):
    """Threshold-free (AUPRC/AUROC) + threshold-0 segment confusion metrics.
    AUPRC/AUROC are the matched-pair metrics vs honest; the rest mirror the
    accuracy/sensitivity/specificity headline the leaky papers report."""
    y_pred = (scores >= 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "roc_auc":       roc_auc_score(y_true, scores),
        "auprc":         average_precision_score(y_true, scores),
        "baseline_auprc": float(y_true.mean()),
        "seg_accuracy":   accuracy_score(y_true, y_pred),
        "seg_specificity": spec,
        "seg_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "seg_precision":  precision_score(y_true, y_pred, zero_division=0),
        "seg_f1":         f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [ ]:
# =====================================================
# CONFIG
# =====================================================
SEED          = 42
PATIENTS      = [f"chb{i:02d}" for i in range(1, 25)]   # chb01 - chb24
USE_ALL_FILES = False
TEST_SIZE     = 0.10


CONFIGS = [
    {"name": "A10", "C": 0.1, "gamma": "scale", "percentile":2 },
    {"name": "B10", "C": 0.1, "gamma": 0.01,   "percentile": 2},
    {"name": "C10", "C": 0.1, "gamma": 0.1,    "percentile": 2},
    {"name": "D10", "C": 1,   "gamma": "scale", "percentile": 2},
    {"name": "E10", "C": 1,   "gamma": 0.01,   "percentile": 2},
    {"name": "F10", "C": 1,   "gamma": 0.1,    "percentile": 2},
    {"name": "G10", "C": 10,  "gamma": "scale", "percentile": 2},
    {"name": "H10", "C": 10,  "gamma": 0.01,   "percentile": 2},
    {"name": "I10", "C": 10,  "gamma": 0.1,    "percentile": 2},
]


OUT_CSV = f"LEAKY_RESULTS_{'all_files' if USE_ALL_FILES else 'seizure_only'}.csv"


# =====================================================
# RUN ALL PATIENTS
# =====================================================

rows = []

for patient in PATIENTS:

    print(f"\n{'='*60}")
    print(f"{patient}  (USE_ALL_FILES={USE_ALL_FILES})")
    print(f"{'='*60}")

    try:
        seizure_records, non_records = load_patient(patient)

        if not seizure_records:
            print(f"  no seizure records — skipping")
            continue


        X_all, y_all = build_pool(
            seizure_records,
            non_records
        )

        print(
            f"  pool windows={X_all.shape[0]} "
            f"class counts={np.bincount(y_all)}"
        )


        Xtr, Xte, ytr, yte = train_test_split(
            X_all,
            y_all,
            test_size=TEST_SIZE,
            random_state=SEED,
            stratify=y_all
        )


        print(
            f"  train/test={len(ytr)}/{len(yte)} "
            f"test positives={int(yte.sum())}"
        )


        for cfg in CONFIGS:

            model = make_pipe(cfg).fit(
                Xtr,
                ytr
            )

            scores = model.decision_function(Xte)

            m = segment_metrics(
                yte,
                scores
            )


            rows.append({

                "config": cfg["name"],
                "C": cfg["C"],
                "gamma": cfg["gamma"],
                "percentile": cfg["percentile"],

                "patient": patient,
                "use_all_files": USE_ALL_FILES,

                "n_train": int(len(ytr)),
                "n_test": int(len(yte)),
                "test_pos": int(yte.sum()),

                **m
            })


            print(
                f"  {cfg['name']}: "
                f"ROC={m['roc_auc']:.3f} "
                f"AUPRC={m['auprc']:.3f} "
                f"SENS={m['seg_sensitivity']:.3f} "
                f"SPEC={m['seg_specificity']:.3f}"
            )


        # SAVE AFTER EACH PATIENT
        checkpoint = pd.DataFrame(rows)

        checkpoint.to_csv(
            OUT_CSV,
            index=False
        )

        print(
            f"  saved checkpoint -> {OUT_CSV} "
            f"({len(checkpoint)} rows)"
        )


    except Exception as e:

        print(f"FAILED {patient}: {e}")

        # still save progress
        pd.DataFrame(rows).to_csv(
            OUT_CSV,
            index=False
        )


# FINAL SAVE

results = pd.DataFrame(rows)

results.to_csv(
    OUT_CSV,
    index=False
)


print("\nDONE")
print(f"Wrote {OUT_CSV}")
print(f"Total rows: {len(results)}")


results


chb01  (USE_ALL_FILES=False)
[SEIZURE]     chb01_03_seizure (1437, 1530) positives = 16
[SEIZURE]     chb01_04_seizure (1437, 1530) positives = 11
[SEIZURE]     chb01_15_seizure (1437, 1530) positives = 16
[SEIZURE]     chb01_16_seizure (1437, 1530) positives = 21
[SEIZURE]     chb01_18_seizure (1437, 1530) positives = 37
[SEIZURE]     chb01_21_seizure (1437, 1530) positives = 38
[SEIZURE]     chb01_26_seizure (927, 1530) positives = 41
[NON-SEIZURE] chb01_01_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_02_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_05_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_06_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_07_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_08_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_09_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_10_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_11_features (1437, 1530) positives = 0
[NON-SEIZURE] chb01_12_feat

## Pooled view per config

Mean segment AUPRC / AUROC per config across patients — the number that pairs
1:1 against the honest run's segment AUPRC for the same config. Compare
segment-to-segment only; never leaky-segment vs honest-event.

In [ ]:
if len(results):
    pooled = (results.groupby(["config", "C", "gamma"])
              [["roc_auc", "auprc", "baseline_auprc"]]
              .mean().reset_index())
    print(pooled.to_string(index=False))

config    C gamma  roc_auc    auprc  baseline_auprc
   A10  0.1 scale 0.992835 0.869048        0.021472
   B10  0.1  0.01 0.992835 0.859524        0.021472
   C10  0.1   0.1 0.951187 0.440642        0.021472
   D10  1.0 scale 0.998209 0.948052        0.021472
   E10  1.0  0.01 0.998657 0.957143        0.021472
   F10  1.0   0.1 0.958352 0.483557        0.021472
   G10 10.0 scale 1.000000 1.000000        0.021472
   H10 10.0  0.01 0.999552 0.982143        0.021472
   I10 10.0   0.1 0.957904 0.482846        0.021472


# With more files

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, accuracy_score, confusion_matrix,
)

In [ ]:
N_FEATURES = 1530
IMAG_TOL   = 1e-6   # |imag| above this = invalid GC window, dropped (as in honest run)

SEIZURE_DIR     = Path("../Features/Seizure")
NON_SEIZURE_DIR = Path("../Features/Non_Seizure")


def _read_feats(h, key="all_feats"):
    """Read features, handling MATLAB complex compound dtype. Returns
    (X real, rows=windows) and a bad-window mask (|imag| too large)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    """Return seizure_records and non_records dicts: {record_name: (X, y)}."""
    records = {}

    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        with h5py.File(f, "r") as h:
            X, bad = _read_feats(h, "all_feats")
            yv = h["all_labels"][:].squeeze()
        y = (yv if yv.shape[0] == X.shape[0] else yv[: X.shape[0]]).astype(int)
        if bad.any():
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)
        print("[SEIZURE]    ", f.stem, X.shape, "positives =", int(y.sum()))

    for f in sorted(NON_SEIZURE_DIR.glob(f"{patient}*_features.mat")):
        with h5py.File(f, "r") as h:
            X, bad = _read_feats(h, "all_feats")
        if bad.any():
            X = X[~bad]
        y = np.zeros(X.shape[0], dtype=int)
        records[f.stem.replace("_features", "")] = (X, y)
        print("[NON-SEIZURE]", f.stem, X.shape, "positives = 0")

    seizure_records = {k: v for k, v in records.items() if v[1].sum() > 0}
    non_records     = {k: v for k, v in records.items() if v[1].sum() == 0}
    return seizure_records, non_records

In [ ]:
def make_pipe(cfg):
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif, percentile=cfg["percentile"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight='balanced', cache_size=500)),
    ])

In [ ]:
def build_pool(seizure_records, non_records):
    """
    Patient-specific window pool.

    Seizure files always included.
    If USE_ALL_FILES:
        include non-seizure files but randomly undersample to MAX_NONSEIZ_WINDOWS.
    """

    rng = np.random.default_rng(42)

    X_parts = []
    y_parts = []


    # ----------------------
    # Seizure windows
    # ----------------------
    for X, y in seizure_records.values():
        X_parts.append(X)
        y_parts.append(y)


    # ----------------------
    # Non-seizure windows
    # ----------------------
    if USE_ALL_FILES:

        X_non = np.vstack(
            [X for X, _ in non_records.values()]
        )

        y_non = np.zeros(
            X_non.shape[0],
            dtype=int
        )


        # undersample negatives
        if X_non.shape[0] > MAX_NONSEIZ_WINDOWS:

            idx = rng.choice(
                X_non.shape[0],
                size=MAX_NONSEIZ_WINDOWS,
                replace=False
            )

            X_non = X_non[idx]
            y_non = y_non[idx]


        X_parts.append(X_non)
        y_parts.append(y_non)


    X = np.vstack(X_parts)
    y = np.hstack(y_parts).astype(int)


    print(
        "POOL:",
        X.shape,
        "seizure windows:",
        int(y.sum()),
        "non-seizure windows:",
        int((y==0).sum())
    )

    return X, y

def segment_metrics(y_true, scores):
    """Threshold-free (AUPRC/AUROC) + threshold-0 segment confusion metrics.
    AUPRC/AUROC are the matched-pair metrics vs honest; the rest mirror the
    accuracy/sensitivity/specificity headline the leaky papers report."""
    y_pred = (scores >= 0).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    return {
        "roc_auc":       roc_auc_score(y_true, scores),
        "auprc":         average_precision_score(y_true, scores),
        "baseline_auprc": float(y_true.mean()),
        "seg_accuracy":   accuracy_score(y_true, y_pred),
        "seg_specificity": spec,
        "seg_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "seg_precision":  precision_score(y_true, y_pred, zero_division=0),
        "seg_f1":         f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


In [ ]:
# =====================================================
# CONFIG
# =====================================================
SEED          = 42
PATIENTS      = [f"chb{i:02d}" for i in range(1, 25)]   # chb01 - chb24
MAX_NONSEIZ_WINDOWS = 10000
USE_ALL_FILES = True
TEST_SIZE     = 0.10


CONFIGS = [
    {"name": "A10", "C": 0.1, "gamma": "scale", "percentile":2 },
    {"name": "B10", "C": 0.1, "gamma": 0.01,   "percentile": 2},
    {"name": "C10", "C": 0.1, "gamma": 0.1,    "percentile": 2},
    {"name": "D10", "C": 1,   "gamma": "scale", "percentile": 2},
    {"name": "E10", "C": 1,   "gamma": 0.01,   "percentile": 2},
    {"name": "F10", "C": 1,   "gamma": 0.1,    "percentile": 2},
    {"name": "G10", "C": 10,  "gamma": "scale", "percentile": 2},
    {"name": "H10", "C": 10,  "gamma": 0.01,   "percentile": 2},
    {"name": "I10", "C": 10,  "gamma": 0.1,    "percentile": 2},
]


OUT_CSV = f"LEAKY_RESULTS_{'all_files' if USE_ALL_FILES else 'seizure_only'}.csv"


# =====================================================
# RUN ALL PATIENTS
# =====================================================

rows = []

for patient in PATIENTS:

    print(f"\n{'='*60}")
    print(f"{patient}  (USE_ALL_FILES={USE_ALL_FILES})")
    print(f"{'='*60}")

    try:
        seizure_records, non_records = load_patient(patient)

        if not seizure_records:
            print(f"  no seizure records — skipping")
            continue


        X_all, y_all = build_pool(
            seizure_records,
            non_records
        )

        print(
            f"  pool windows={X_all.shape[0]} "
            f"class counts={np.bincount(y_all)}"
        )


        Xtr, Xte, ytr, yte = train_test_split(
            X_all,
            y_all,
            test_size=TEST_SIZE,
            random_state=SEED,
            stratify=y_all
        )


        print(
            f"  train/test={len(ytr)}/{len(yte)} "
            f"test positives={int(yte.sum())}"
        )


        for cfg in CONFIGS:

            model = make_pipe(cfg).fit(
                Xtr,
                ytr
            )

            scores = model.decision_function(Xte)

            m = segment_metrics(
                yte,
                scores
            )


            rows.append({

                "config": cfg["name"],
                "C": cfg["C"],
                "gamma": cfg["gamma"],
                "percentile": cfg["percentile"],

                "patient": patient,
                "use_all_files": USE_ALL_FILES,

                "n_train": int(len(ytr)),
                "n_test": int(len(yte)),
                "test_pos": int(yte.sum()),

                **m
            })


            print(
                f"  {cfg['name']}: "
                f"ROC={m['roc_auc']:.3f} "
                f"AUPRC={m['auprc']:.3f} "
                f"SENS={m['seg_sensitivity']:.3f} "
                f"SPEC={m['seg_specificity']:.3f}"
            )


        # SAVE AFTER EACH PATIENT
        checkpoint = pd.DataFrame(rows)

        checkpoint.to_csv(
            OUT_CSV,
            index=False
        )

        print(
            f"  saved checkpoint -> {OUT_CSV} "
            f"({len(checkpoint)} rows)"
        )


    except Exception as e:

        print(f"FAILED {patient}: {e}")

        # still save progress
        pd.DataFrame(rows).to_csv(
            OUT_CSV,
            index=False
        )


# FINAL SAVE

results = pd.DataFrame(rows)

results.to_csv(
    OUT_CSV,
    index=False
)


print("\nDONE")
print(f"Wrote {OUT_CSV}")
print(f"Total rows: {len(results)}")


results